# 01 — Data Pipeline: Feast Feature Engineering

**Prerequisite:** Raw data must already be in S3 (run the data-download Job manifest first).

```bash
# One-time: download HuggingFace → MinIO (see manifests/data-download-job.yaml)
envsubst < manifests/data-download-job.yaml | oc apply -f -
oc logs -f job/smartshop-data-download-medium -n smartshop
```

## How it works

Feast `@batch_feature_view` UDFs define PySpark transformations (groupBy, agg, join)
that run inside the **SparkComputeEngine** with RAPIDS GPU acceleration during
`feast materialize`. No separate SparkApplication or intermediate S3 parquets needed.

```
S3 raw reviews + metadata → feast materialize (Spark + RAPIDS) → Redis
```

- **user_features**: per-user aggregates (avg rating, review count, tenure, etc.)
- **item_features**: per-item aggregates + metadata join (title, brand, category, price)

**Runs from:** RHOAI Workbench (in-cluster)


In [11]:
%pip install -q kubernetes boto3 tabulate pandas pyarrow feast redis pyyaml s3fs yamlmagic
%load_ext yamlmagic


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
The yamlmagic extension is already loaded. To reload it, use:
  %reload_ext yamlmagic


## Parameters

Edit these to customize the data pipeline run.


In [12]:
%%yaml parameters

CATEGORIES: "Electronics,Books,Home_and_Kitchen"
MATERIALIZE_END: "2026-12-31T23:59:59"


<IPython.core.display.Javascript object>

In [13]:
from _config import *
globals().update(parameters)

validate()
print(f"Categories:   {CATEGORIES}")
print(f"Materialize:  {MATERIALIZE_START} → {MATERIALIZE_END}")

Namespace:  smartshop
S3:         http://minio.smartshop.svc.cluster.local:9000
Redis:      redis.smartshop.svc.cluster.local:6379
Milvus:     milvus.smartshop.svc.cluster.local:19530
Feast:      feast-smartshop-feast-registry.smartshop.svc.cluster.local:443
Config OK ✓
Categories:   Electronics,Books,Home_and_Kitchen
Materialize:  2020-01-01T00:00:00 → 2026-12-31T23:59:59


---
## Pre-flight: Verify Raw Data in S3

Confirms that raw reviews and metadata are already loaded in MinIO.
If empty, run the data-download manifest first (see header).


In [14]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"Existing buckets: {buckets}")

# --- Raw data (input) ---
resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=5)
raw_files = resp.get("Contents", [])
if raw_files:
    print(f"\n✓ Raw reviews: {len(raw_files)}+ files in smartshop-raw/raw/reviews/")
else:
    print("\n✗ No raw data — run the download manifest first:")
    print("  envsubst < manifests/data-download-job.yaml | oc apply -f -")
    raise RuntimeError("Raw data missing")

resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/metadata/", MaxKeys=3)
meta_files = resp.get("Contents", [])
print(f"{'✓' if meta_files else '✗'} Metadata: {len(meta_files)}+ files in smartshop-raw/raw/metadata/")

# --- Feature data (output of Phase 1) ---
for prefix, label in [("user_features/", "User features"), ("item_features/", "Item features")]:
    r2 = s3.list_objects_v2(Bucket="smartshop-features", Prefix=prefix, MaxKeys=1)
    has = r2.get("KeyCount", 0) > 0
    print(f"{'⚠ exists' if has else '○ empty ':8s} smartshop-features/{prefix}")

if not meta_files:
    print("\n⚠ Metadata missing — feature engineering will produce items without title/brand/category")

Existing buckets: ['milvus', 'smartshop-embeddings', 'smartshop-features', 'smartshop-models', 'smartshop-raw', 'smartshop-spark-logs']

✓ Raw reviews: 5+ files in smartshop-raw/raw/reviews/
✓ Metadata: 3+ files in smartshop-raw/raw/metadata/
○ empty  smartshop-features/user_features/
○ empty  smartshop-features/item_features/


## 0.1 Verify Raw Data in S3


In [15]:
from tabulate import tabulate

print("=== Reviews ===\n")
paginator = s3.get_paginator("list_objects_v2")
total_size = 0
total_files = 0
rows = []

for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/reviews/"):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        total_size += obj["Size"]
        total_files += 1
        if total_files <= 15:
            rows.append([obj["Key"], f"{size_mb:.1f} MB"])

try:
    print(tabulate(rows, headers=["Key", "Size"], tablefmt="simple"))
except:
    for r in rows:
        print(f"  {r[0]:60s} {r[1]}")

if total_files > 15:
    print(f"  ... and {total_files - 15} more files")
print(f"\nTotal: {total_files} files, {total_size / (1024**3):.2f} GB")

print("\n=== Metadata ===\n")
meta_files = 0
meta_size = 0
for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/metadata/"):
    for obj in page.get("Contents", []):
        meta_files += 1
        meta_size += obj["Size"]
        if meta_files <= 10:
            print(f"  {obj['Key']:60s} {obj['Size'] / (1024*1024):.1f} MB")
print(f"\nTotal metadata: {meta_files} files, {meta_size / (1024**3):.2f} GB")

=== Reviews ===

Key                                  Size
-----------------------------------  --------
raw/reviews/Books/part-0000.parquet  158.8 MB
raw/reviews/Books/part-0001.parquet  142.0 MB
raw/reviews/Books/part-0002.parquet  117.9 MB
raw/reviews/Books/part-0003.parquet  135.4 MB
raw/reviews/Books/part-0004.parquet  156.7 MB
raw/reviews/Books/part-0005.parquet  178.2 MB
raw/reviews/Books/part-0006.parquet  125.3 MB
raw/reviews/Books/part-0007.parquet  115.7 MB
raw/reviews/Books/part-0008.parquet  176.2 MB
raw/reviews/Books/part-0009.parquet  182.1 MB
raw/reviews/Books/part-0010.parquet  189.4 MB
raw/reviews/Books/part-0011.parquet  144.2 MB
raw/reviews/Books/part-0012.parquet  149.3 MB
raw/reviews/Books/part-0013.parquet  129.9 MB
raw/reviews/Books/part-0014.parquet  128.7 MB
  ... and 267 more files

Total: 282 files, 27.57 GB

=== Metadata ===

  raw/metadata/Electronics_meta.parquet                        47.4 MB
  raw/metadata/Electronics_meta/part-0000.parquet             

In [16]:
import pandas as pd
import io

# Quick peek at the first review file
resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=1)
if resp.get("Contents"):
    first_key = resp["Contents"][0]["Key"]
    obj = s3.get_object(Bucket="smartshop-raw", Key=first_key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"Preview: {first_key}")
    print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
    print()
    display(df.head(3)) if hasattr(__builtins__, '__IPYTHON__') else print(df.head(3).to_string())
else:
    print("No review files found — download may have failed.")

Preview: raw/reviews/Books/part-0000.parquet
Rows: 500,000 | Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']



,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Not a watercolor book! Seems like copies imo.,It is definitely not a watercolor book. The p...,[{'small_image_url': 'https://m.media-amazon.c...,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1642399598485,0,True
1,5.0,Updated: after 1st arrived damaged this one is...,Updated: after first book arrived very damaged...,[],0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640629604904,1,True
2,5.0,Excellent! I love it!,I bought it for the bag on the front so it pai...,[],1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640383495102,0,True


## 0.2 S3 Bucket Overview


In [17]:
import s3fs

fs = s3fs.S3FileSystem(
    key=AWS_KEY,
    secret=AWS_SECRET,
    client_kwargs={"endpoint_url": S3_ENDPOINT},
)

for bucket in ["smartshop-raw", "smartshop-features", "smartshop-models"]:
    try:
        files = fs.ls(bucket)
        print(f"\n{bucket}/ ({len(files)} top-level items)")
        for f in files[:8]:
            print(f"  {f}")
        if len(files) > 8:
            print(f"  ... and {len(files) - 8} more")
    except Exception:
        print(f"\n{bucket}/ (not found or empty)")



smartshop-raw/ (3 top-level items)
  smartshop-raw/jars
  smartshop-raw/processed
  smartshop-raw/raw

smartshop-features/ (2 top-level items)
  smartshop-features/llm_data
  smartshop-features/spark-events

smartshop-models/ (5 top-level items)
  smartshop-models/llm-adapter
  smartshop-models/llm-checkpoints
  smartshop-models/rec-checkpoints
  smartshop-models/recommendation
  smartshop-models/workspaces


---
# Phase 1: Feast Feature Engineering + Materialization (RAPIDS GPU)

Feast `@batch_feature_view` UDFs define the feature engineering logic inline.
`feast materialize` triggers the **SparkComputeEngine** (with RAPIDS GPU) to:
1. Read raw reviews + metadata from S3
2. Apply PySpark transformations (groupBy, agg, join)
3. Write computed features directly to Redis

No separate SparkApplication or intermediate S3 parquets needed.

## 1.1 Connect to Kubernetes & Find Feast Pod


In [ ]:
import warnings, urllib3, time, base64
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

from kubernetes import client as k8s_client, config as k8s_config
from kubernetes.stream import stream
k8s_config.load_incluster_config()
v1 = k8s_client.CoreV1Api()

FEAST_REPO = FEAST_FEATURE_REPO_ON_POD

feast_pods = v1.list_namespaced_pod(
    NAMESPACE, label_selector="feast.dev/name=smartshop-feast"
).items
feast_pod = next(p.metadata.name for p in feast_pods if p.status.phase == "Running")
print(f"Feast pod: {feast_pod}")

# Inject S3 credentials for Spark's s3a connector
secret = v1.read_namespaced_secret("smartshop-credentials", NAMESPACE)
aws_key = base64.b64decode(secret.data["AWS_ACCESS_KEY_ID"]).decode()
aws_secret = base64.b64decode(secret.data["AWS_SECRET_ACCESS_KEY"]).decode()


def feast_exec(cmd, timeout=600):
    """Run a command in the Feast offline container, streaming output."""
    full_cmd = f"export AWS_ACCESS_KEY_ID={aws_key} AWS_SECRET_ACCESS_KEY={aws_secret} && {cmd}"
    resp = stream(
        v1.connect_get_namespaced_pod_exec,
        feast_pod, NAMESPACE, container="offline",
        command=["bash", "-c", full_cmd],
        stderr=True, stdout=True, stdin=False, tty=False,
        _preload_content=True,
    )
    return resp


print("\n--- feast apply ---")
out = feast_exec(f"feast -c {FEAST_REPO} apply 2>&1")
print(out)

## 1.2 Trigger Feast Materialization

Runs `feast materialize` which executes the `@batch_feature_view` UDFs via SparkComputeEngine.
This reads raw S3 data, computes user/item features, and writes to Redis in one step.


In [ ]:
import redis

import yaml as _yaml

redis_secret = v1.read_namespaced_secret("feast-redis-secret", NAMESPACE)
redis_blob = _yaml.safe_load(base64.b64decode(redis_secret.data["redis"]).decode())
redis_conn = redis_blob.get("connection_string", "")
redis_pw = dict(kv.split("=", 1) for kv in redis_conn.split(",") if "=" in kv).get("password", "")
r = redis.Redis(host=REDIS_HOST.split("//")[-1].split(":")[0], port=6379, password=redis_pw)

keys_before = r.dbsize()
print(f"Redis BEFORE: {keys_before:,} keys")

print(f"\n--- feast materialize (user_features + item_features) ---")
print(f"Time range: 2020-01-01 → {MATERIALIZE_END}")
print("This reads raw S3 data, runs PySpark UDFs with RAPIDS, writes to Redis...\n")

mat_cmd = f"feast -c {FEAST_REPO} materialize 2020-01-01T00:00:00 {MATERIALIZE_END} -v user_features -v item_features"
out = feast_exec(mat_cmd, timeout=3600)

for line in out.strip().split("\n")[-20:]:
    print(line)

keys_after = r.dbsize()
print(f"\nRedis AFTER: {keys_after:,} keys (new: {keys_after - keys_before:,})")

## 1.3 Monitor Spark Executor Pods

Run this cell during materialization to see Spark executor pod status.


In [ ]:
executor_pods = v1.list_namespaced_pod(
    NAMESPACE, label_selector="spark-role=executor",
)

if executor_pods.items:
    print(f"Spark executor pods: {len(executor_pods.items)}")
    for p in executor_pods.items:
        phase = p.status.phase
        gpu = "GPU" if any("gpu" in str(c.resources) for c in p.spec.containers) else "CPU"
        print(f"  {p.metadata.name}: {phase} ({gpu})")
else:
    print("No executor pods (materialization may be running in local[*] mode)")

---
# Phase 2: Verify Features in Redis

## 2.1 Verify with Feast SDK — `get_online_features()`

Query the Redis online store to confirm user and item features are materialized.


In [ ]:
import tempfile, shutil, os
from feast import FeatureStore

# --- Feast registry (via mounted client config) ---
config_dir = tempfile.mkdtemp(prefix="feast-nb-")
shutil.copy(FEAST_CLIENT_CONFIG, os.path.join(config_dir, "feature_store.yaml"))
store = FeatureStore(repo_path=config_dir)
print("Feast registry connected ✓")
fvs = store.list_feature_views()
for fv in fvs:
    features = [f.name for f in fv.features] if hasattr(fv, "features") else []
    print(f"  {fv.name:25s} ({len(features)} features)")

# --- Redis stats ---
# (cell kept minimal — Redis was already checked in Phase 1)
redis_secret = v1.read_namespaced_secret("feast-redis-secret", NAMESPACE)
redis_conn_yaml = base64.b64decode(redis_secret.data["redis"]).decode()
redis_pw = ""
for line in redis_conn_yaml.split("\n"):
    if "password=" in line:
        redis_pw = line.split("password=")[1].strip().strip('"')

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=redis_pw or None, decode_responses=False)
keys_before = r.dbsize()
print(f"Redis BEFORE flush: {keys_before:,} keys")

r.flushdb()
print(f"Redis flushed ✓ — {r.dbsize()} keys")

# --- Feast registry ---
config_dir = tempfile.mkdtemp(prefix="feast-nb-")
shutil.copy("/opt/app-root/src/feast-config/smartshop", os.path.join(config_dir, "feature_store.yaml"))
store = FeatureStore(repo_path=config_dir)
print(f"\nFeast registry connected ✓")
fvs = store.list_feature_views()
for fv in fvs:
    features = [f.name for f in fv.features] if hasattr(fv, 'features') else []
    print(f"  {fv.name:25s} ({len(features)} features)")

# --- Find Feast pod ---
pods = v1.list_namespaced_pod(NAMESPACE, label_selector="feast.dev/name=smartshop-feast", field_selector="status.phase=Running")
feast_pod = pods.items[0].metadata.name if pods.items else None
if not feast_pod:
    for p in v1.list_namespaced_pod(NAMESPACE).items:
        if "feast-smartshop" in p.metadata.name and p.status.phase == "Running":
            feast_pod = p.metadata.name
            break
assert feast_pod, "No running Feast pod found!"
print(f"\nFeast pod: {feast_pod}")

## 2.2 Verify with Feast SDK — `get_online_features()`

Query the Redis online store to confirm user and item features are materialized.


In [ ]:
import pandas as pd

# Discover real user IDs from Redis — single SCAN call, parse Feast v3 binary key format
import struct

_, raw_keys = r.scan(cursor=0, match=b"*user_id*smartshop", count=20)
user_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    marker = b"user_id"
    pos = raw.find(marker)
    if pos < 0:
        continue
    # After entity name: \x02\x00\x00\x00 (type tag) + 4-byte little-endian length + value + "smartshop"
    offset = pos + len(marker) + 4  # skip type tag
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        user_ids.append(val)

sample_users = [{"user_id": uid} for uid in user_ids[:5]]
print(f"Querying {len(sample_users)} real user IDs: {[u['user_id'] for u in sample_users]}\n")

result = store.get_online_features(
    features=[
        "user_features:user_avg_rating",
        "user_features:user_review_count",
        "user_features:user_unique_items",
        "user_features:user_tenure_days",
    ],
    entity_rows=sample_users,
).to_dict()

df = pd.DataFrame(result)
print("User features from Redis:")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

In [ ]:
# Discover real item IDs (ASINs) from Redis — single SCAN call
_, raw_keys = r.scan(cursor=0, match=b"*item_id*smartshop", count=20)
item_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    marker = b"item_id"
    pos = raw.find(marker)
    if pos < 0:
        continue
    offset = pos + len(marker) + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        item_ids.append(val)

sample_items = [{"item_id": iid} for iid in item_ids[:5]]
print(f"Querying {len(sample_items)} real item IDs (ASINs): {[i['item_id'] for i in sample_items]}\n")

result = store.get_online_features(
    features=[
        "item_features:item_title",
        "item_features:item_brand",
        "item_features:item_category",
        "item_features:item_avg_rating",
        "item_features:item_price",
        "item_features:item_review_count",
    ],
    entity_rows=sample_items,
).to_dict()

df = pd.DataFrame(result)
print("Item features from Redis (with product metadata):")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

---
## Summary

| Phase | Step | What happened |
|-------|------|---------------|
| **0** | Pre-flight | Verified raw data in S3 (reviews + metadata) |
| **1** | Feature Engineering + Materialization | `feast materialize` with `@batch_feature_view` UDFs — raw S3 → Redis (Spark + RAPIDS GPU) |
| **2** | Validation | `get_online_features()` confirmed user/item features + product metadata in Redis |

**Key RHOAI capabilities shown:**
- Feast `@batch_feature_view` with PySpark UDFs for feature engineering
- SparkComputeEngine with NVIDIA RAPIDS GPU acceleration
- Single `feast materialize` command: raw data → computed features → Redis
- Product metadata enrichment (title, brand, category, price) via inline join

**Next:** Run `02_training.ipynb` to train the recommendation model and fine-tune the LLM.
